# 02 - Predicting the missing precursor charge

About 1.5 % of the cleaned spectra carry no precursor-charge annotation. A CatBoost classifier
is fitted on the annotated spectra and used to fill the gaps, restricted to the three charge
states retained by notebook 01.

The notebook is organised as setup, feature construction, model definition and evaluation
protocol, hyperparameter search, final training, a comparison against the recorded history of
earlier runs, and a disabled section that performs the actual imputation. Earlier exploratory
feature sets have been removed; their scores remain available in
`model_training_metadata_precursor_charge.json`.


In [4]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

import refac_common as c

RANDOM_STATE = 1312
N_FOLDS = 3
SEARCH_N = 40_000        # stratified subsample the hyperparameter search runs on
N_TRIALS = 30
MAX_ITERATIONS = 3_000   # upper bound; the effective count comes from early stopping
EARLY_STOPPING = 100
RUN_SEARCH = True        # False reuses the cached best parameters
RUN_IMPUTATION = False

METADATA_PATH = Path("model_training_metadata_precursor_charge.json")
STUDY_PATH = Path("optuna_precursor_charge.db")
BEST_PARAMS_PATH = Path("best_params_precursor_charge.json")


In [5]:
# clean_spectra_test_split.parquet is deliberately never loaded in this notebook.
train = pd.read_parquet("clean_spectra_train_split.parquet")
val = pd.read_parquet("clean_spectra_val_split.parquet")
unknown = pd.read_parquet("data_without_precursor_charge.parquet")

train["precursor_charge"] = train["precursor_charge"].astype(int)
val["precursor_charge"] = val["precursor_charge"].astype(int)
assert set(train["precursor_charge"]) == set(c.CLASSES)
assert set(val["precursor_charge"]) == set(c.CLASSES)
assert unknown["precursor_charge"].isna().all()

known = pd.concat([train, val], ignore_index=True)

print("train:", len(train), "| validation:", len(val), "| to impute:", len(unknown))
known["precursor_charge"].value_counts().sort_index()


train: 279832 | validation: 34979 | to impute: 3728


precursor_charge
2    192434
3    112547
4      9830
Name: count, dtype: int64

## Experiment bookkeeping

Every configuration evaluated below is recorded in a single, continuously appended JSON file,
`model_training_metadata_precursor_charge.json`.  
The file holds a list of data provenance, the evaluation protocol and one entry per configuration with its parameters, feature list and scores. 

interrupting is possible,

Saved models are named `precursor_catboost_<timestamp>_f1_<score>.cbm`


In [6]:
RUN_STARTED = datetime.now().astimezone()
RUN_KEY = RUN_STARTED.strftime("%Y-%m-%d %H:%M:%S %z")

run_record = {
    "notebook": "02_precursor_charge.ipynb",
    "started_at": RUN_STARTED.isoformat(),
    "random_state": RANDOM_STATE,
    "classes": list(c.CLASSES),
    "data": {
        "train_rows": len(train),
        "val_rows": len(val),
        "unknown_rows": len(unknown),
        "search_subsample": SEARCH_N,
        "cv": f"StratifiedGroupKFold(n_splits={N_FOLDS}, groups=stripped_sequence)",
    },
    "configurations": [],
    "artifacts": {},
}


def write_run_metadata():
    """Append this run to the continuous metadata file, replacing an earlier write of the same run."""
    entries = json.loads(METADATA_PATH.read_text()) if METADATA_PATH.exists() else []
    entries = [e for e in entries if RUN_KEY not in e]
    entries.append({RUN_KEY: run_record})
    METADATA_PATH.write_text(json.dumps(entries, indent=2, default=str))
    return entries


def log_config(name, params, metrics, **extra):
    run_record["configurations"].append({
        "name": name,
        "logged_at": datetime.now().astimezone().isoformat(),
        "model": "CatBoostClassifier",
        "hyperparameters": params,
        "n_features": len(FEATURE_COLUMNS),
        "features": FEATURE_COLUMNS,
        "metrics": metrics,
        **extra,
    })
    entries = write_run_metadata()
    print(f"logged '{name}' -> {METADATA_PATH} ({len(entries)} runs, "
          f"{len(run_record['configurations'])} configs this run)")


def model_path(score):
    return Path(f"precursor_catboost_{RUN_STARTED:%Y%m%d-%H%M%S}_f1_{score:.4f}.cbm")


## Features
#### new features
Main feature classes are:
- **Proton competition**: 
- **Missed cleavages**:
- **Size**
- **Spectrum shape**
- **Proline**

In [7]:
FEATURE_COLUMNS = [
    # size
    "length", "pep_mass", "mass_per_len",
    # composition and proton competition
    "n_KRH", "n_DE", "n_K", "n_R", "n_H", "n_P",
    "basic_density", "acid_density", "p_density",
    "weighted_basicity", "weighted_basic_density", "isoelectric_point", "gravy_index",
    "n_ionizable", "n_ionizable_capped", "ionizable_density", "len_per_ionizable",
    # cleavage context
    "n_internal_KR", "nterm_basic", "cterm_basic", "cterm_is_K", "cterm_is_R",
    # fragment series
    "n_b", "n_y", "max_b", "max_y", "frag_coverage", "max_frag_rel",
    "by_ratio", "by_int_ratio", "y_completeness",
    # intensity distribution
    "int_mean", "int_std", "int_max", "int_median", "int_sum",
    "int_entropy", "int_top3_frac",
]


def _sequence_features(seqs: pd.Index) -> pd.DataFrame:
    """Sequence-derived features for a set of distinct peptides."""
    stripped = seqs.str.replace(r"\[UNIMOD:\d+\]", "", regex=True)
    L = stripped.str.len().to_numpy()
    cnt = lambda aa: stripped.str.count(aa).to_numpy()

    n_K, n_R, n_H, n_P = cnt("K"), cnt("R"), cnt("H"), cnt("P")
    n_basic = n_K + n_R + n_H
    n_acid = cnt("D") + cnt("E")

    mass = np.full(len(seqs), c.WATER_MASS)
    for aa, m in c.AA_MONO_MASS.items():
        mass += cnt(aa) * m
    for tok, m in c.MOD_MASS.items():
        mass += np.fromiter((s.count(tok) for s in seqs), np.int64, len(seqs)) * m

    pi = np.empty(len(seqs))
    gravy = np.empty(len(seqs))
    for i, s in enumerate(stripped):
        try:
            analysis = ProteinAnalysis(s)
            pi[i], gravy[i] = analysis.isoelectric_point(), analysis.gravy()
        except ValueError:
            pi[i], gravy[i] = 7.0, 0.0  # neutral fallback for ambiguous residues

    weighted_basicity = n_K * 10.53 + n_R * 12.48 + n_H * 6.00
    n_ionizable = n_basic + 1  # basic side chains plus the free N-terminus
    head = stripped.str[:-1]
    cterm = stripped.str[-1]

    return pd.DataFrame({
        "length": L,
        "pep_mass": mass,
        "mass_per_len": mass / L,
        "n_KRH": n_basic,
        "n_DE": n_acid,
        "n_K": n_K,
        "n_R": n_R,
        "n_H": n_H,
        "n_P": n_P,
        "basic_density": n_basic / L,
        "acid_density": n_acid / L,
        "p_density": n_P / L,
        "weighted_basicity": weighted_basicity,
        "weighted_basic_density": weighted_basicity / L,
        "isoelectric_point": pi,
        "gravy_index": gravy,
        "n_ionizable": n_ionizable,
        "n_ionizable_capped": np.minimum(n_ionizable, max(c.CLASSES)),
        "ionizable_density": n_ionizable / L,
        "len_per_ionizable": L / n_ionizable,
        "n_internal_KR": (head.str.count("K") + head.str.count("R")).to_numpy(),
        "nterm_basic": stripped.str[0].isin(c.AA_BASIC).astype(np.int64),
        "cterm_basic": cterm.isin(c.AA_BASIC).astype(np.int64),
        "cterm_is_K": (cterm == "K").astype(np.int64),
        "cterm_is_R": (cterm == "R").astype(np.int64),
    }, index=seqs)


_seq_cache: pd.DataFrame | None = None


def sequence_table(df: pd.DataFrame) -> pd.DataFrame:
    """Cached sequence features; the frames featurised below are nested, so peptides recur."""
    global _seq_cache
    seqs = pd.Index(pd.unique(df["peptide_sequence"]), name="peptide_sequence")
    missing = seqs if _seq_cache is None else seqs.difference(_seq_cache.index)
    if len(missing):
        new = _sequence_features(missing)
        _seq_cache = new if _seq_cache is None else pd.concat([_seq_cache, new])
    return _seq_cache


def _spectrum_features(df: pd.DataFrame) -> pd.DataFrame:
    """Per-spectrum ion and intensity statistics, computed over the flattened ragged arrays."""
    n = len(df)
    lens = np.fromiter((len(a) for a in df["intensities_raw"]), np.int64, n)
    assert lens.min() > 0, "spectrum without matched peaks"
    ion_lens = np.fromiter((len(a) for a in df["matched_ions"]), np.int64, n)
    assert (ion_lens == lens).all(), "ion labels and intensities are misaligned"

    raw = np.concatenate(df["intensities_raw"].to_list()).astype(np.float64)
    flat = np.log1p(raw)
    ends = np.cumsum(lens)
    starts = ends - lens
    row = np.repeat(np.arange(n), lens)

    total = np.add.reduceat(flat, starts)
    mean = total / lens
    var = np.add.reduceat((flat - mean[row]) ** 2, starts) / lens

    out = {
        "int_mean": mean,
        "int_std": np.sqrt(var),
        "int_max": np.maximum.reduceat(flat, starts),
        "int_median": np.fromiter(
            (np.median(flat[s:e]) for s, e in zip(starts, ends)), np.float64, n),
        "int_sum": total,
    }

    ions = pd.Index(np.concatenate(df["matched_ions"].to_list()))
    kind = ions.str[0].to_numpy()
    num = ions.str[1:].astype(np.int64).to_numpy()
    for tag in ("b", "y"):
        m = kind == tag
        out[f"n_{tag}"] = np.bincount(row[m], minlength=n)
        largest = np.zeros(n, dtype=np.int64)
        np.maximum.at(largest, row[m], num[m])
        out[f"max_{tag}"] = largest
        out[f"{tag}_int_sum"] = np.bincount(row[m], weights=raw[m], minlength=n)

    y = kind == "y"
    out["n_y_unique"] = (pd.Series(num[y]).groupby(row[y]).nunique()
                         .reindex(np.arange(n), fill_value=0).to_numpy())

    raw_sum = np.add.reduceat(raw, starts)
    share = raw / np.repeat(np.where(raw_sum > 0, raw_sum, 1.0), lens)
    log_share = np.zeros_like(share)
    np.log(share, out=log_share, where=share > 0)
    out["int_entropy"] = np.bincount(row, weights=-share * log_share, minlength=n)
    top3 = np.fromiter((np.sort(raw[s:e])[-3:].sum() for s, e in zip(starts, ends)), np.float64, n)
    out["int_top3_frac"] = top3 / np.where(raw_sum > 0, raw_sum, np.nan)

    return pd.DataFrame(out)


def make_X(df: pd.DataFrame) -> pd.DataFrame:
    seq = (sequence_table(df)
           .reindex(df["peptide_sequence"].to_numpy())
           .reset_index(drop=True))
    X = pd.concat([seq, _spectrum_features(df)], axis=1)

    L = X["length"].to_numpy()
    X["frag_coverage"] = (X["n_b"] + X["n_y"]) / np.maximum(2 * (L - 1), 1)
    X["max_frag_rel"] = np.maximum(X["max_b"], X["max_y"]) / L
    X["by_ratio"] = X["n_b"] / X["n_y"].replace(0, np.nan)
    X["y_completeness"] = X["n_y_unique"] / np.maximum(L - 1, 1)
    by_total = X["b_int_sum"] + X["y_int_sum"]
    X["by_int_ratio"] = np.where(by_total > 0, X["b_int_sum"] / by_total, np.nan)
    return X[FEATURE_COLUMNS]


## Model and evaluation protocol

A CatBoost multiclass classifier over the three retained charge states. Scores are macro-F1,
which weights charge 4 — roughly 3 % of the annotated spectra — equally with the dominant
charge 2, and accuracy as a secondary figure.

Folds come from `StratifiedGroupKFold` with the stripped peptide sequence as the grouping key,
so that near-identical spectra of one peptide cannot straddle a fold boundary; stratification
keeps charge 4 represented in every fold. Within a fold, a further grouped 15 % of the training
rows is held out purely to drive early stopping, which keeps the evaluation fold untouched by
the stopping decision.


In [8]:
def make_model(params, **overrides):
    return CatBoostClassifier(
        loss_function="MultiClass",
        class_names=list(c.CLASSES),
        random_state=RANDOM_STATE,
        thread_count=-1,
        verbose=0,
        **{**params, **overrides},
    )


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", labels=list(c.CLASSES))


def grouped_folds(df, y, n_splits=N_FOLDS):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    return list(sgkf.split(df, y, groups=df["stripped_sequence"].to_numpy()))


def early_stopping_split(idx, groups, size=0.15):
    gss = GroupShuffleSplit(n_splits=1, test_size=size, random_state=RANDOM_STATE)
    fit, es = next(gss.split(idx, groups=groups[idx]))
    return idx[fit], idx[es]


## Hyperparameter search ( enabled !! )

We search via a Tree-structured parzen Estimator over depth, learning rate, L2 regularisation, random stretch, bagging temperature, split granularity and class weighing scheme.  
As we are mostly improving macro F1 because of our 20:1 imbalance between charge 2 and 4, the weighing scheme is important and schould not be fixed by hand.
To not burn half the amazon rainforest (or bavarian blackforest) in the forest we:
1. Do not search amount of boosting iterations - we run against a large cap and start early
2. Trials report after every fold and are cut by successive-half pruning (unpromising configs get evicted after one fold and not a full run)
3. Running on a stratified subsample, while only fitting on the full as soon as we select a config
4. Finally we persist to disk so we can start where we left off

In [9]:
search_df = train.sample(min(SEARCH_N, len(train)), random_state=RANDOM_STATE).reset_index(drop=True)
y_search = search_df["precursor_charge"].to_numpy()
groups_search = search_df["stripped_sequence"].to_numpy()
X_search = make_X(search_df)          # computed once, reused by every trial
folds_search = grouped_folds(search_df, y_search)


def suggest_params(trial):
    return {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_categorical("border_count", [32, 64, 128]),
        "auto_class_weights": trial.suggest_categorical(
            "auto_class_weights", [None, "Balanced", "SqrtBalanced"]),
    }


def objective(trial):
    params = suggest_params(trial)
    scores, iters = [], []
    for k, (tr, te) in enumerate(folds_search):
        fit_idx, es_idx = early_stopping_split(tr, groups_search)
        model = make_model(
            params,
            iterations=MAX_ITERATIONS,
            boosting_type="Plain",
            early_stopping_rounds=EARLY_STOPPING,
            use_best_model=True,
        )
        model.fit(X_search.iloc[fit_idx], y_search[fit_idx],
                  eval_set=(X_search.iloc[es_idx], y_search[es_idx]))
        pred = model.predict(X_search.iloc[te]).ravel().astype(int)
        scores.append(macro_f1(y_search[te], pred))
        iters.append(int(model.best_iteration_ or MAX_ITERATIONS))

        trial.report(float(np.mean(scores)), k)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("mean_best_iteration", int(np.mean(iters)))
    return float(np.mean(scores))


In [10]:
if RUN_SEARCH:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        study_name="precursor_charge",
        storage=f"sqlite:///{STUDY_PATH}",
        load_if_exists=True,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=RANDOM_STATE, multivariate=True, n_startup_trials=10),
        pruner=optuna.pruners.SuccessiveHalvingPruner(),
    )
    study.optimize(objective, n_trials=N_TRIALS)
    states = pd.Series([t.state.name for t in study.trials]).value_counts().to_dict()
    best = {
        "params": study.best_params,
        "iterations": int(study.best_trial.user_attrs["mean_best_iteration"]),
        "cv_macro_f1": float(study.best_value),
        "trials": states,
    }
    BEST_PARAMS_PATH.write_text(json.dumps(best, indent=2))
else:
    best = json.loads(BEST_PARAMS_PATH.read_text())

print(f"best CV macro-F1 = {best['cv_macro_f1']:.4f} over {best['trials']}")
print(json.dumps(best["params"], indent=2))

log_config(
    "search_best",
    dict(best["params"], iterations=best["iterations"]),
    {"cv_macro_f1_mean": best["cv_macro_f1"]},
    search={"n_rows": len(search_df), "n_folds": N_FOLDS, "trials": best["trials"]},
)


[W 2026-09-25 01:13:22,705] Trial 24 failed with parameters: {'depth': 8, 'learning_rate': 0.1454379561413177, 'l2_leaf_reg': 13.519052853017612, 'random_strength': 7.488140978146734, 'bagging_temperature': 0.9932706551684013, 'border_count': 128, 'auto_class_weights': None} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "/Users/fridolin.karger/dev/sommer/.venv/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/fc/y1fk3zyx12q75f3nh6n9rn300000gp/T/ipykernel_30476/3657463784.py", line 33, in objective
    model.fit(X_search.iloc[fit_idx], y_search[fit_idx],
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
              eval_set=(X_search.iloc[es_idx], y_search[es_idx]))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/fridolin.karger/dev/sommer/.venv/lib/python3.14/site-packages/catboost/core.py", line 5547, in fit


KeyboardInterrupt: 

## Final training

The winning configuration is refitted on the full training split

In [ ]:
final_params = dict(best["params"], iterations=int(best["iterations"] * 1.1))

X_train, y_train = make_X(train), train["precursor_charge"].to_numpy()
X_val, y_val = make_X(val), val["precursor_charge"].to_numpy()

model = make_model(final_params)
model.fit(X_train, y_train)
val_pred = model.predict(X_val).ravel().astype(int)
VAL_F1 = macro_f1(y_val, val_pred)

print(classification_report(y_val, val_pred, labels=list(c.CLASSES)))
importances = (pd.Series(model.get_feature_importance(), index=FEATURE_COLUMNS)
               .sort_values(ascending=False))
print(importances.head(12).round(2).to_string())

log_config(
    "tuned_val_holdout",
    final_params,
    {
        "val_macro_f1": float(VAL_F1),
        "val_accuracy": float(accuracy_score(y_val, val_pred)),
        "per_class": classification_report(y_val, val_pred, labels=list(c.CLASSES),
                                           output_dict=True),
        "feature_importance": {k: float(v) for k, v in importances.items()},
    },
    fitted_on="train split",
    evaluated_on="val split",
)

# Artefact: same configuration, refitted on every annotated spectrum.
full_model = make_model(final_params)
full_model.fit(make_X(known), known["precursor_charge"].to_numpy())

MODEL_PATH = model_path(VAL_F1)
full_model.save_model(str(MODEL_PATH))
run_record["artifacts"]["model"] = {
    "path": str(MODEL_PATH),
    "format": "catboost cbm",
    "saved_at": datetime.now().astimezone().isoformat(),
    "load_with": f"CatBoostClassifier().load_model('{MODEL_PATH}')",
    "val_macro_f1": float(VAL_F1),
    "hyperparameters": final_params,
    "features": FEATURE_COLUMNS,
    "class_names": [int(v) for v in full_model.classes_],
    "fitted_on": "train + val splits (annotated spectra)",
    "n_fit_rows": len(known),
}
write_run_metadata()
print("saved model ->", MODEL_PATH)


## Comparison against earlier runs


In [ ]:
rows = []
for entry in json.loads(METADATA_PATH.read_text()):
    for stamp, rec in entry.items():
        for cfg in rec["configurations"]:
            metrics = cfg["metrics"]
            metric = "val" if "val_macro_f1" in metrics else "cv"
            score = metrics.get("val_macro_f1", metrics.get("cv_macro_f1_mean"))
            if score is not None:
                rows.append({
                    "run": stamp,
                    "config": cfg["name"],
                    "metric": metric,
                    "n_features": cfg["n_features"],
                    "macro_f1": round(score, 4),
                    "current": stamp == RUN_KEY,
                })

history = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
print(history.to_string(index=False))

previous = history[~history["current"]]["macro_f1"]
if len(previous):
    print(f"\ncurrent val macro-F1 {VAL_F1:.4f} vs best previously recorded "
          f"{previous.max():.4f} ({VAL_F1 - previous.max():+.4f})")


## Imputation of the full dataset (disabled)


In [ ]:
OUT_PATH = Path("spectra_with_charge.parquet")

if RUN_IMPUTATION:
    X_unknown = make_X(unknown)
    proba = full_model.predict_proba(X_unknown)
    imputed = np.asarray(full_model.classes_, dtype=int)[proba.argmax(axis=1)]

    known_out = known.assign(charge_source="annotated", charge_confidence=1.0)
    unknown_out = unknown.assign(
        precursor_charge=imputed,
        charge_source="predicted",
        charge_confidence=proba.max(axis=1),
    )
    full = pd.concat([known_out, unknown_out], ignore_index=True)
    full["precursor_charge"] = full["precursor_charge"].astype(int)
    assert set(full["precursor_charge"]) <= set(c.CLASSES)
    assert full.loc[full["charge_source"] == "predicted", "split"].isna().all()
    full.to_parquet(OUT_PATH, index=False)

    print(pd.Series(imputed).value_counts().sort_index().to_string())
    print("wrote", OUT_PATH, full.shape)

    run_record["artifacts"]["spectra_with_charge"] = {
        "path": str(OUT_PATH),
        "saved_at": datetime.now().astimezone().isoformat(),
        "rows": int(len(full)),
        "imputed_distribution": {int(k): int(v) for k, v in
                                 pd.Series(imputed).value_counts().items()},
        "median_confidence": float(np.median(proba.max(axis=1))),
    }
else:
    print(f"imputation skipped (RUN_IMPUTATION is False); {len(unknown):,} spectra left unannotated")

run_record["finished_at"] = datetime.now().astimezone().isoformat()
write_run_metadata()


'          macro_f1  macro_f1_std  accuracy  macro_f1_delta\nbaseline    0.7648        0.0398    0.9846          0.0000\nimproved    0.7959        0.0088    0.9877          0.0311'